<a href="https://colab.research.google.com/github/jm5155/PYTHON/blob/main/cleaning2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np

np.random.seed(42)


n = 120

data = {
    "student_id": [f"S{1000+i}" for i in range(n)],
    "program": np.random.choice(["MIT", "MSCS", "MSDS"], n),
    "age": np.random.randint(22, 50, n),
    "gender": np.random.choice(["Male", "Female"], n),
    "attendance_rate": np.random.normal(85, 10, n).round(2),
    "quiz_average": np.random.normal(82, 12, n).round(2),
    "assignment_score": np.random.normal(80, 15, n).round(2),
    "final_exam_score": np.random.normal(78, 14, n).round(2),
    "lms_login_count": np.random.randint(5, 80, n)
}

df = pd.DataFrame(data)


df.loc[[5, 12, 25, 40], "age"] = np.nan
df.loc[[8, 19, 33, 61], "quiz_average"] = np.nan
df.loc[[10, 35, 70], "final_exam_score"] = np.nan
df.loc[[15, 44], "program"] = np.nan


df.loc[[3, 18, 27], "program"] = "M.I.T."
df.loc[[6, 22], "program"] = "Masters in IT"
df.loc[[9, 30], "program"] = "mscs"
df.loc[[11, 45], "gender"] = "female"
df.loc[[20, 50], "gender"] = "M"


duplicates = df.iloc[[2, 14, 29]]
df = pd.concat([df, duplicates], ignore_index=True)


df.loc[7, "attendance_rate"] = 150
df.loc[13, "quiz_average"] = -10
df.loc[21, "assignment_score"] = 250
df.loc[31, "lms_login_count"] = 500


df = df.sample(frac=1).reset_index(drop=True)

df.head()


print("Shape of Dataset:")
print(df.shape)

print("\nDataset Information:")
df.info()

print("\nSummary Statistics:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Records:")
print(df.duplicated().sum())


issues = pd.DataFrame({
    "Issue Type": [
        "Missing Value",
        "Missing Value",
        "Missing Value",
        "Missing Value",
        "Inconsistency",
        "Inconsistency",
        "Outlier",
        "Outlier",
        "Outlier",
        "Outlier",
        "Duplicate"
    ],
    "Column Affected": [
        "age",
        "program",
        "quiz_average",
        "final_exam_score",
        "program",
        "gender",
        "attendance_rate",
        "quiz_average",
        "assignment_score",
        "lms_login_count",
        "student_id"
    ],
    "Example Found": [
        "NaN",
        "NaN",
        "NaN",
        "NaN",
        "M.I.T., Masters in IT, mscs",
        "female, M",
        "150",
        "-10",
        "250",
        "500",
        "Repeated student record"
    ],
    "Possible Cause": [
        "Incomplete student profile",
        "Missing program entry",
        "Missing quiz score",
        "Missing exam score",
        "Different encoding",
        "Different capitalization",
        "Encoding error",
        "Encoding error",
        "Encoding error",
        "System error",
        "Duplicate export"
    ]
})

issues


df_clean = df.copy()

df_clean["age"] = df_clean["age"].fillna(df_clean["age"].median())
df_clean["quiz_average"] = df_clean["quiz_average"].fillna(df_clean["quiz_average"].median())
df_clean["final_exam_score"] = df_clean["final_exam_score"].fillna(df_clean["final_exam_score"].median())

df_clean["program"] = df_clean["program"].fillna(df_clean["program"].mode()[0])

print(df_clean.isnull().sum())


print("Duplicate Records:")
print(df_clean[df_clean.duplicated()])

df_clean = df_clean.drop_duplicates()

print("\nRemaining Duplicates:")
print(df_clean.duplicated().sum())


program_map = {
    "M.I.T.": "MIT",
    "Masters in IT": "MIT",
    "mscs": "MSCS"
}

gender_map = {
    "female": "Female",
    "M": "Male"
}

df_clean["program"] = df_clean["program"].replace(program_map)
df_clean["gender"] = df_clean["gender"].replace(gender_map)

print(df_clean["program"].value_counts())
print(df_clean["gender"].value_counts())


df_clean.loc[df_clean["attendance_rate"] > 100, "attendance_rate"] = np.nan
df_clean.loc[df_clean["attendance_rate"] < 0, "attendance_rate"] = np.nan

df_clean.loc[df_clean["quiz_average"] > 100, "quiz_average"] = np.nan
df_clean.loc[df_clean["quiz_average"] < 0, "quiz_average"] = np.nan

df_clean.loc[df_clean["assignment_score"] > 100, "assignment_score"] = np.nan
df_clean.loc[df_clean["assignment_score"] < 0, "assignment_score"] = np.nan

df_clean.loc[df_clean["final_exam_score"] > 100, "final_exam_score"] = np.nan
df_clean.loc[df_clean["final_exam_score"] < 0, "final_exam_score"] = np.nan

df_clean.loc[df_clean["lms_login_count"] < 0, "lms_login_count"] = np.nan
df_clean.loc[df_clean["lms_login_count"] > 200, "lms_login_count"] = np.nan

df_clean["attendance_rate"] = df_clean["attendance_rate"].fillna(df_clean["attendance_rate"].median())
df_clean["quiz_average"] = df_clean["quiz_average"].fillna(df_clean["quiz_average"].median())
df_clean["assignment_score"] = df_clean["assignment_score"].fillna(df_clean["assignment_score"].median())
df_clean["final_exam_score"] = df_clean["final_exam_score"].fillna(df_clean["final_exam_score"].median())
df_clean["lms_login_count"] = df_clean["lms_login_count"].fillna(df_clean["lms_login_count"].median())

df_clean.describe()


print("Missing Values")
print(df_clean.isnull().sum())

print("\nDuplicate Records")
print(df_clean.duplicated().sum())

print("\nProgram Counts")
print(df_clean["program"].value_counts())

print("\nGender Counts")
print(df_clean["gender"].value_counts())

print("\nAttendance Valid:", df_clean["attendance_rate"].between(0,100).all())
print("Quiz Valid:", df_clean["quiz_average"].between(0,100).all())
print("Assignment Valid:", df_clean["assignment_score"].between(0,100).all())
print("Final Exam Valid:", df_clean["final_exam_score"].between(0,100).all())


comparison = pd.DataFrame({
    "Original Missing Values": df.isnull().sum(),
    "Cleaned Missing Values": df_clean.isnull().sum()
})

comparison


print("Original Dataset Summary")
print(df.describe())

print("Cleaned Dataset Summary")
print(df_clean.describe())


cleaning_log = pd.DataFrame({
    "Column": [
        "age",
        "program",
        "quiz_average",
        "final_exam_score",
        "attendance_rate",
        "assignment_score",
        "lms_login_count",
        "student_id"
    ],
    "Issue Found": [
        "Missing values",
        "Inconsistent labels",
        "Missing values",
        "Missing values",
        "Value above 100",
        "Value above 100",
        "Extreme value",
        "Duplicate records"
    ],
    "Cleaning Action": [
        "Filled with median",
        "Standardized values",
        "Filled with median",
        "Filled with median",
        "Replaced then imputed",
        "Replaced then imputed",
        "Replaced then imputed",
        "Removed duplicates"
    ],
    "Reason / Justification": [
        "Median is less affected by outliers.",
        "Ensures consistent grouping.",
        "Median preserves the distribution.",
        "Median is robust against outliers.",
        "Attendance cannot exceed 100.",
        "Assignment scores cannot exceed 100.",
        "500 logins is unrealistic.",
        "Duplicate records should not be counted twice."
    ]
})

cleaning_log




Shape of Dataset:
(123, 9)

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 123 entries, 0 to 122
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        123 non-null    object 
 1   program           121 non-null    object 
 2   age               119 non-null    float64
 3   gender            123 non-null    object 
 4   attendance_rate   123 non-null    float64
 5   quiz_average      119 non-null    float64
 6   assignment_score  123 non-null    float64
 7   final_exam_score  120 non-null    float64
 8   lms_login_count   123 non-null    int64  
dtypes: float64(5), int64(1), object(3)
memory usage: 8.8+ KB

Summary Statistics:
              age  attendance_rate  quiz_average  assignment_score  \
count  119.000000       123.000000    119.000000        123.000000   
mean    35.016807        85.809675     79.722101         81.361789   
std      9.031943        12.278603   

,Column,Issue Found,Cleaning Action,Reason / Justification
0,age,Missing values,Filled with median,Median is less affected by outliers.
1,program,Inconsistent labels,Standardized values,Ensures consistent grouping.
2,quiz_average,Missing values,Filled with median,Median preserves the distribution.
3,final_exam_score,Missing values,Filled with median,Median is robust against outliers.
4,attendance_rate,Value above 100,Replaced then imputed,Attendance cannot exceed 100.
5,assignment_score,Value above 100,Replaced then imputed,Assignment scores cannot exceed 100.
6,lms_login_count,Extreme value,Replaced then imputed,500 logins is unrealistic.
7,student_id,Duplicate records,Removed duplicates,Duplicate records should not be counted twice.
